In [ ]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

# Set absolute path for new metastore
new_metastore_path = os.path.abspath('/apps/sandbox/metastore')
metastore_url = f"jdbc:derby:;databaseName={new_metastore_path};create=true"

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .set("spark.sql.catalogImplementation", "hive")
    .set("spark.sql.catalog.spark_catalog.warehouse","s3a://warehouse/default")
    .set("spark.sql.warehouse.dir", "s3a://warehouse/default/spark")
    .set("javax.jdo.option.ConnectionURL", metastore_url)
    .set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.797,io.delta:delta-spark_2.12:3.3.2")
    .set("spark.executor.heartbeatInterval", "300000")
    .set("spark.network.timeout", "400000")
    .set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    .set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    .set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.delta.enableFastS3AListFrom", "true")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

# configure the SparkSession with the configure_spark_with_delta_pip() utility function in Delta Lake:
builder = SparkSession.builder.appName("04-parquet-to-delta").master("local[*]").config(conf=sparkConf)
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4"]).getOrCreate()

#
spark.sparkContext.setLogLevel('ERROR')
spark

#
# 
#
def display(df):
    df.show(truncate=False)

Ivy Default Cache set to: /home/brijeshdhaker/.ivy2/cache
The jars for the packages stored in: /home/brijeshdhaker/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ad1ee80f-a7c0-4e33-910b-5fcae3f6639e;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/spark-3.5.3/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.3.2 in local-m2-cache
	found io.delta#delta-storage;3.3.2 in local-m2-cache
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
	found org.apache.hadoop#hadoop-aws;3.3.4 in local-m2-cache
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in local-m2-cache
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
:: resolution report :: resolve 172ms :: artifacts dl 8ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from local-m2-cache in [default]
	io.delta#delta-spark_2.12;3.3.2 from local-m2-cache in [default]
	io.delta#delta-storage;3.3.2 from local-m2-cache in [default]
	org.antlr#antlr4-runtime;4.9.3 from local-m2-cache in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from local-m2-cache in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from local-m2-cache in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   arti

In [2]:
%%bash

## Delete Existing Delta Table /invoices
aws --endpoint-url http://minio.sandbox.net:9010 s3 ls s3://warehouse/default/deltalake --recursive

2026-04-01 09:59:37       1064 default/deltalake/delta_table/_delta_log/00000000000000000000.crc
2026-04-01 09:59:37       1259 default/deltalake/delta_table/_delta_log/00000000000000000000.json
2026-04-01 10:15:20       1804 default/deltalake/delta_table/_delta_log/00000000000000000001.crc
2026-04-01 10:15:19       1117 default/deltalake/delta_table/_delta_log/00000000000000000001.json
2026-04-01 10:40:06       1814 default/deltalake/delta_table/_delta_log/00000000000000000002.crc
2026-04-01 10:40:06       1594 default/deltalake/delta_table/_delta_log/00000000000000000002.json
2026-04-01 10:40:08       1812 default/deltalake/delta_table/_delta_log/00000000000000000003.crc
2026-04-01 10:40:08       1597 default/deltalake/delta_table/_delta_log/00000000000000000003.json
2026-04-01 10:40:28       1812 default/deltalake/delta_table/_delta_log/00000000000000000004.crc
2026-04-01 10:40:28       1613 default/deltalake/delta_table/_delta_log/00000000000000000004.json
2026-04-01 10:40:30      

In [5]:
%load_ext sql

In [6]:
%sql spark

In [3]:

df = spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet")

df.write.mode("overwrite").parquet("s3a://warehouse/default/deltalake/invoices_1_100_v1")

df.write.mode("overwrite").parquet("s3a://warehouse/default/deltalake/invoices_1_100_v2")

In [7]:
%%sql

CONVERT TO DELTA PARQUET.`s3a://warehouse/default/deltalake/invoices_1_100_v1`;

Running query in 'SparkSession'

++
||
++
++

In [8]:
from delta.tables import DeltaTable
DeltaTable.convertToDelta(
    spark,
    "parquet.`s3a://warehouse/default/deltalake/invoices_1_100_v2`"
)

In [0]:
# df = spark.read.csv(path)
# df.write.mode("overwrite").format("delta").save(path)

In [9]:
%%sql

CREATE OR REPLACE TABLE spark_catalog.deltalake.invoices_ext
USING DELTA
LOCATION 's3a://warehouse/default/deltalake/invoices_ext'
AS SELECT * FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`

Running query in 'SparkSession'

++
||
++
++

In [0]:
%%sql

DROP TABLE spark_catalog.deltalake.invoices_ext;

In [ ]:
%%sql

SHOW TABLES IN spark_catalog.deltalake;

UsageError: Cell magic `%%sql` not found.
